In [1]:
import numpy as np
import pandas as pd
#import cv2
import os
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
%matplotlib qt
import os
from os import listdir
from os.path import isfile, join
import re

In [3]:
def PSDanalysis(list2,list3,t):
    S=max(list3)[0]+min(list3)[0]
    #print(S)
    D=max(list3)[0]-min(list3)[0]
    #print(D
    list4=[]
    count=0
    for i in range(len(list2)):
        if ((2*list2[i][0])-S)/D >1:
            #print(i)
            #print(((2*list2[i][0])-S)/D)
            list4.append(1)
            count+=1
        if ((2*list2[i][0])-S)/D <-1:
            #print(i)
            #print(((2*list2[i][0])-S)/D)
            list4.append(-1)
            count+=1
        else:
            list4.append(((2*list2[i][0])-S)/D)
    fs = len(list4)/t
    f, Pxx_den = signal.welch(list4, fs, window='hann',nperseg=1e6)
    print(count)
    return(f, Pxx_den)

In [5]:
def AnalyseHalf(maindir, fandir, time, mainname):
    main=np.genfromtxt(maindir,skip_header=5,usecols=1 ,delimiter=',', names=True, case_sensitive=True).tolist()
    if fandir=='self':
        fan=main
    else:
        fan=np.genfromtxt(fandir,skip_header=5,usecols=1 ,delimiter=',', names=True, case_sensitive=True).tolist()
    main_f, main_Pxx = PSDanalysis(main,fan,time)
    np.save(mainname+'_f.npy',main_f)
    np.save(mainname+'_Pxx.npy',main_Pxx)
    return(main_f, main_Pxx)

In [30]:
def AnalyseFull(maindir, fandir, time):
    main = np.genfromtxt(maindir, skip_header=5, usecols=1, delimiter=',', names=True, case_sensitive=True).tolist()
    if fandir == 'self':
        fan = main
    else:
        fan = np.genfromtxt(fandir, skip_header=5, usecols=1, delimiter=',', names=True, case_sensitive=True).tolist()
    date_match = re.search(r'(\d{4}-\d{2}-\d{2})', maindir)
    if date_match:
        date_str = date_match.group(1)
    else:
        raise ValueError(f"Could not extract date from path: {maindir}")
    filename = os.path.basename(maindir)
    match = re.search(r'_([a-zA-Z0-9]+)_([a-zA-Z0-9]+)--(\d+)\.csv$', filename)
    if match:
        filetype = match.group(1)      # e.g., detectornoise
        config = match.group(2)        # e.g., 5ms
        index = int(match.group(3))    # strip leading zeros
        shortname = f"{date_str}_{filetype}_{config}_{index}"
    else:
        raise ValueError(f"Could not parse shortname from filename: {filename}")
    main_f, main_Pxx = PSDanalysis(main, fan, time)
    np.save(f"{shortname}_f.npy", main_f)
    np.save(f"{shortname}_Pxx.npy", main_Pxx)
    return main_f, main_Pxx

In [9]:
def fildirs(mypath):
    onlyfiles = [os.path.join(mypath, f) for f in os.listdir(mypath) if 
    os.path.isfile(os.path.join(mypath, f))]
    return(onlyfiles)

In [48]:
mypath=r'C:\Users\Ry\Documents\PhD Thesis Writing\Data\2025-07-17\Balanced'

In [50]:
refpath=fildirs(r'C:\Users\Ry\Documents\PhD Thesis Writing\Data\2025-07-17\Reference')

In [52]:
refpath[0]

'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Reference\\C1--fibrefibre_link_balanced_5ms--00000.csv'

In [54]:
if 'fan' in refpath:
    print('found')
else:
    print('not found')

not found


In [56]:
balanceddirs=fildirs(mypath)

In [58]:
balanceddirs

['C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_250ks--00000.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_250ks--00001.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_500ms--00000.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_500ms--00001.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_50ms--00000.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_50ms--00001.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_5gs--00000.csv',
 'C:\\Users\\Ry\\Documents\\PhD Thesis Writing\\Data\\2025-07-17\\Balanced\\C1--fibrefibre_link_balanced_5gs--00001.csv',
 'C:\\Users\\R

In [40]:
param_map = {
    '5ms': {'time': 5},
    '50ms': {'time': 0.5},
    '500ms': {'time': 0.1},
    '5gs': {'time': 0.01},
    '250ks': {'time': 20}
}

In [42]:
def AnalyseDirs(file_paths,fandir,param_map):
    pattern = re.compile(r'_([a-zA-Z0-9]+)--')
    for path in file_paths:
        filename = os.path.basename(path)
        match = pattern.search(filename)
        if match:
            key = match.group(1)
            if key in param_map:
                params = param_map[key]
                AnalyseFull(path, fandir, **params)
            else:
                print(f"Unknown key '{key}' in file {filename}, skipping.")
        else:
            print(f"Could not extract parameters from {filename}")

In [60]:
AnalyseDirs(balanceddirs,refpath[0],param_map)

0
0
6
3
1
0
0
1
1


In [62]:
mypath2=r'C:\Users\Ry\Documents\PhD Thesis Writing\Data\2025-07-17\Balanced'

In [64]:
unbalanceddirs=fildirs(mypath2)

In [66]:
AnalyseDirs(unbalanceddirs,refpath[0],param_map)

0
0
6
3
1
0
0
1
1


In [50]:
detnoisedir=fildirs(r'G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\Detector Noise')

In [53]:
print(detnoisedir)

['G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\Detector Noise\\C1--fibrefs_detectornoise_5ms--00000.csv', 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\Detector Noise\\C1--fibrefs_detectornoise_500ms--00000.csv', 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\Detector Noise\\C1--fibrefs_detectornoise_250ks--00000.csv', 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\Detector Noise\\C1--fibrefs_detectornoise_50ms--00000.csv']


In [25]:
print(fsmoffdir[0])
print(fsmoffdir[1])
print(fsmoffdir[2])
print(fsmoffdir[3])
print(fsmoffdir[4])

G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMOFF\C1--fibrefs_fsmoff_5ms--00001.csv
G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMOFF\C1--fibrefs_fsmoff_250ks--00002.csv
G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMOFF\C1--fibrefs_fsmoff_250ks--00001.csv
G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMOFF\C1--fibrefs_fsmoff_5ms--00000.csv
G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMOFF\C1--fibrefs_fsmoff_250ks--00000.csv


In [84]:
AnalyseHalf(fsmoffdir[8], fsmoffdir[0], 20, 'FSMOFF_250ks_2')

15


(array([0.00000000e+00, 1.25000005e+00, 2.50000010e+00, ...,
        6.24997525e+05, 6.24998775e+05, 6.25000025e+05]),
 array([2.80728501e-04, 1.32614009e-03, 1.58537437e-03, ...,
        6.69461766e-08, 4.81120792e-07, 7.57458959e-07]))

In [86]:
AnalyseHalf(fsmoffdir[11], fsmoffdir[0], 5, 'FSMOFF_5ms_1')
AnalyseHalf(fsmoffdir[4], fsmoffdir[0], 0.5, 'FSMOFF_50ms_1')
AnalyseHalf(fsmoffdir[3], fsmoffdir[0], 0.1, 'FSMOFF_500ms_1')

0
0
0


(array([0.00000000e+00, 5.00000010e+02, 1.00000002e+03, ...,
        2.49999005e+08, 2.49999505e+08, 2.50000005e+08]),
 array([5.19532816e-08, 1.50889041e-06, 1.39374588e-07, ...,
        2.98347182e-10, 4.05179134e-09, 7.20844452e-09]))

In [88]:
OFF_250ks_f=np.load('FSMOFF_250ks_2_f.npy')
OFF_250ks_Pxx=np.load('FSMOFF_250ks_2_Pxx.npy')

In [89]:
OFF_5ms_f=np.load('FSMOFF_5ms_1_f.npy')
OFF_5ms_Pxx=np.load('FSMOFF_5ms_1_Pxx.npy')

In [90]:
OFF_50ms_f=np.load('FSMOFF_50ms_1_f.npy')
OFF_50ms_Pxx=np.load('FSMOFF_50ms_1_Pxx.npy')

In [91]:
OFF_500ms_f=np.load('FSMOFF_500ms_1_f.npy')
OFF_500ms_Pxx=np.load('FSMOFF_500ms_1_Pxx.npy')

In [93]:
plt.loglog(OFF_250ks_f,OFF_250ks_Pxx)
plt.loglog(OFF_5ms_f,OFF_5ms_Pxx)
plt.loglog(OFF_50ms_f,OFF_50ms_Pxx)
plt.loglog(OFF_500ms_f,OFF_500ms_Pxx)

In [94]:
mypath2=r'G:\Shared drives\QComms Research\QComms Research\People\RR\Data\21_07_2025\FSMON'

In [95]:
fsmondir=fildirs(mypath2)

In [96]:
fsmondir

['G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_250ks--00000.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_250ks--00001.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_5ms--00000.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_fan--00001.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_500ms--00000.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_500ms--00002.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibrefs_fsmon_500ms--00001.csv',
 'G:\\Shared drives\\QComms Research\\QComms Research\\People\\RR\\Data\\21_07_2025\\FSMON\\C1--fibre

In [107]:
f250,p250=AnalyseHalf(fsmondir[1], fsmondir[3], 20, 'FSMON_250ks_1')
f5,p5=AnalyseHalf(fsmondir[10], fsmondir[3], 5, 'FSMON_5ms_1')
f50,p50=AnalyseHalf(fsmondir[8], fsmondir[3], 0.5, 'FSMON_50ms_1')
f500,p500=AnalyseHalf(fsmondir[6], fsmondir[3], 0.1, 'FSMON_500ms_1')

KeyboardInterrupt: 

In [109]:
plt.loglog(f250,p250)
plt.loglog(f5,p5)
plt.loglog(f50,p50)
plt.loglog(f500,p500)

In [100]:
def SplicePSD4(PSDs, splices):
    xs=[]
    ys=[]
    for i in range(len(PSDs[0][0])):
        if PSDs[0][0][i]<splices[0]:
            xs.append(PSDs[0][0][i])
            ys.append(PSDs[0][1][i])
    for i in range(len(PSDs[1][0])):
        if PSDs[1][0][i]>splices[0]:
            if PSDs[1][0][i]<splices[1]:
                xs.append(PSDs[1][0][i])
                ys.append(PSDs[1][1][i])
    for i in range(len(PSDs[2][0])):
        if PSDs[2][0][i]>splices[1]:
            if PSDs[2][0][i]<splices[2]:
                xs.append(PSDs[2][0][i])
                ys.append(PSDs[2][1][i])
    for i in range(len(PSDs[3][0])):
        if PSDs[3][0][i]>splices[2]:
            xs.append(PSDs[3][0][i])
            ys.append(PSDs[3][1][i])
    return(xs,ys)

In [102]:
splicestest=[61,1000,8500]

In [104]:
PSDstest=[(f250,p250),(f5,p5),(f50,p50),(f500,p500)]

In [105]:
testxs,testys=SplicePSD4(PSDstest,splicestest)

In [110]:
plt.loglog(testxs,testys)

In [111]:
PSDtest2=[(OFF_250ks_f,OFF_250ks_Pxx),(OFF_5ms_f,OFF_5ms_Pxx),(OFF_50ms_f,OFF_50ms_Pxx),(OFF_500ms_f,OFF_500ms_Pxx)]

In [113]:
testxs2,testys2=SplicePSD4(PSDtest2,splicestest)

In [115]:
plt.loglog(testxs,testys, label='FSM ON')
plt.loglog(testxs2,testys2, label='FSM OFF')
plt.legend()